In [ ]:
import os
import sys
import time
import glob
import argparse
import traceback
from ascii import header, footer

import json
import h5py
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D

from scipy.fft import rfft, rfftfreq
from scipy.stats import beta

from PyPDF2 import PdfMerger
from PyPDF2 import PdfReader, PdfWriter

In [ ]:
def main():
    """
    Main entry point for DUNE light file processing and plotting.
    """
    # start timer
    start_time = time.time()

    # print header
    header()

    args = parse_args()
    if args.units not in ['ADC16', 'ADC14', 'V']:
        print(f"Invalid units: {args.units}. Must be 'ADC16', 'ADC14', or 'V'.")
        sys.exit(1)

    if args.powspec_nevts > args.max_evts:
        print("Number of events for the power spectrum can't be greater than the total events")
        # set powerspec_nevts to max_evts
        args.powspec_nevts = args.max_evts
        print(f"Setting powerspec_nevts to {args.max_evts} (beware of memory issues with large nevts)")

    try:
        os.makedirs(args.output_dir, exist_ok=True)
        print(f"Output directory is ready: {args.output_dir}")
    except Exception as e:
        print(f"Error creating output directory {args.output_dir}: {e}")
        raise

    try:
        os.makedirs(args.tmp_dir, exist_ok=True)
        print(f"Temporary directory is ready: {args.tmp_dir}")
    except Exception as e:
        print(f"Error creating temporary directory {args.tmp_dir}: {e}")
        raise

        # search for any files in --input_path starting with args.name_conv and ending with '.FLOW.hdf5'
    all_files = [f for f in os.listdir(args.input_path) if f.startswith(args.file_syntax) and f.endswith('.FLOW.hdf5')]
    all_files.sort()            # HERE BE DRAGONS
    n_available_files = len(all_files)
    print(f"Found {n_available_files} files in {args.input_path} starting with {args.file_syntax} and ending with .FLOW.hdf5")
    if n_available_files == 0:
        print("No files found. Exiting.")
        sys.exit(1)
    if args.start_run >= n_available_files:
        print(f"Start file index {args.start_run} is out of range. There are only {n_available_files} files available. Exiting.")
        sys.exit(1)
    if args.start_run + args.nfiles > n_available_files:
        args.nfiles = n_available_files - args.start_run
        print(f"Adjusting number of files to process to {args.nfiles} to avoid going out of range.")

    file_time = start_time
    proc_files = 0

    for i_file in range(args.start_run, args.start_run + args.nfiles, 1):

        # Construct the filename based on the input path and file index
        # filename = f'{args.input_path}{all_files[i_file]}'
        # HACK: Use start_run literally, instead of as an index into a (wrongly-)sorted list
        # (this is OK for the nearline, which just runs on one file at a time)
        filename = f'{args.input_path}{args.file_syntax}{i_file}.FLOW.hdf5'
        # just the file name, no path and cutting off .FLOW.hdf5
        # short_filename = all_files[i_file][:-10]
        short_filename = os.path.basename(filename)[:-10]

        # Skip if file does not exist
        if not os.path.exists(filename):
            print(f"File not found, skipping: {filename}")
            continue
        proc_files += 1
        print(f"Processing file: {filename} with units: {args.units}")

        ptps = get_ptps(args.units)

        try:
            #### DEFINE NUMBER OF GB TO KEEP IN args.max_gb ####
            file = h5py.File(filename, 'r')
            ## keep opened data below XXX GB to avoid crashes ##
            size_bytes = os.path.getsize(input_file)
            size_gb = size_bytes / (1024 ** args.max_gb)
            print(f"File size: {size_gb:.2f} GB")

            MULT = int(size_gb // args.max_gb)
            if MULT < 1:
                MULT = 1
            print(f"Using MULT = {MULT} for data reduction.")
            print(f"File opened successfully: {filename}")
        except Exception as e:
            print(f"Error opening {filename}: {e}")
            continue

        # Get start and end timestamps in ms
        start_timestamp = file["light/events/data"][0]['utime_ms'][0]
        end_timestamp = file["light/events/data"][-1]['utime_ms'][0]

        # Convert to UTC datetime string
        start_utc = time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(start_timestamp / 1000))
        end_utc = time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(end_timestamp / 1000))

        # Convert to US Central Time (UTC-6)
        start_central = time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(start_timestamp / 1000 - 6 * 3600))
        end_central = time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime(end_timestamp / 1000 - 6 * 3600))
        print(f"File start time (US Central, UTC-6): {start_central}")
        print(f"File end time (US Central, UTC-6): {end_central}")

        # files for comparison
        if args.ncomp == -1 or args.ncomp > i_file:
            ncomps = np.arange(0, i_file, 1)
        elif args.ncomp == 0:
            ncomps = np.array([])
        else:
            ncomps = np.arange(i_file-args.ncomp, i_file, 1)